# Demonstração de funcionamento
## Previsão do resultado de partidas do Brasileirão

**Inteligência Artificial II · AMF · 2026/02**

Execute as células em ordem (*Kernel > Restart & Run All*). O treinamento usa as temporadas 2020–2021, a seleção usa 2022 e o teste usa 2023.
Esta é uma **avaliação retrospectiva**, com informações disponíveis antes de cada partida; o placar real só aparece para conferir a previsão.
Todos os números abaixo são calculados nesta execução.

In [1]:
from pathlib import Path
import sys
raiz = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(raiz))
from src.documentation_evidence import executar_evidencias
dados = executar_evidencias()

INÍCIO | Execução real do protocolo temporal (treino 2020–21, validação 2022, teste 2023)

ETAPA 1: Treinamento em 2020-2021 e avaliação na validação (2022)
[Baseline (Majoritária)  ] Acurácia: 44.35% | Bal Acc: 33.33% | Macro F1: 0.205 | LogLoss: 20.057
[Regressão Logística     ] Acurácia: 41.60% | Bal Acc: 37.42% | Macro F1: 0.370 | LogLoss: 1.122


[Random Forest           ] Acurácia: 43.53% | Bal Acc: 33.92% | Macro F1: 0.257 | LogLoss: 1.059


[HistGradientBoosting    ] Acurácia: 37.19% | Bal Acc: 33.22% | Macro F1: 0.326 | LogLoss: 1.172

ETAPA 2: Modelo selecionado: Regressão Logística
Critério: maior Macro F1 na validação (0.370)

ETAPA 3: Reajuste em 2020 a 2022 (1052 partidas utilizáveis)

ETAPA 4: Avaliação final no teste (temporada 2023)
-> Baseline no teste: acurácia 46.96% | Macro F1 0.213
-> Regressão Logística no teste: acurácia 44.48% | Bal Acc 37.26% | Macro F1 0.360 | LogLoss 1.089



[OK] Modelo salvo em models/model.joblib
[OK] Metadados salvos em models/metadata.json


[OK] Relatório gravado em reports/resultados_modelagem.md
CONCLUÍDO | Regressão Logística | acurácia teste 44.48% | Macro F1 0.360


## 1. Comparação dos algoritmos na validação (2022)
Cada modelo foi ajustado em 2020–2021. A seleção usa o maior Macro F1 (a referência majoritária não participa da escolha).

In [2]:
import pandas as pd
from IPython.display import display, HTML, Markdown
display(HTML("<style>.dataframe{font-size:15px!important}.dataframe th,.dataframe td{padding:8px 10px!important}</style>"))

def pct(v): return f"{v:.2%}"
def num(v): return "—" if v is None else f"{v:.3f}"

linhas = []
for nome, m in dados["modelos"].items():
    tr, va = m["treino"], m["validacao"]
    linhas.append({"Modelo": nome, "Acurácia treino": pct(tr["acuracia"]), "Acurácia validação": pct(va["acuracia"]),
                   "Precisão macro": num(va["precisao_macro"]), "Recall macro": num(va["recall_macro"]),
                   "Macro F1": num(va["f1_macro"]), "Log-Loss": num(va["log_loss"]), "Tempo de ajuste (s)": f'{m["tempo_treino_s"]:.2f}'})
display(pd.DataFrame(linhas).style.hide(axis="index"))
print("Modelo selecionado:", dados["modelo_selecionado"], "| critério:", dados["criterio_selecao"])

Modelo,Acurácia treino,Acurácia validação,Precisão macro,Recall macro,Macro F1,Log-Loss,Tempo de ajuste (s)
Baseline (Majoritária),45.57%,44.35%,0.148,0.333,0.205,20.057,0.01
Regressão Logística,56.31%,41.60%,0.381,0.374,0.370,1.122,0.02
Random Forest,58.49%,43.53%,0.301,0.339,0.257,1.059,0.29
HistGradientBoosting,91.00%,37.19%,0.336,0.332,0.326,1.172,0.31


Modelo selecionado: Regressão Logística | critério: maior Macro F1 na validação (2022), excluindo a referência majoritária


## 2. Avaliação final no teste (2023)
O modelo selecionado foi reajustado com 2020–2022 e avaliado uma única vez em 2023. A referência de frequências históricas usa as proporções de classes do mesmo período de treino.

In [3]:
teste = dados["final"]["teste"]
linhas = []
for nome, r in [(dados["modelo_selecionado"], teste), ("Sempre mandante", dados["baselines_teste"]["majoritaria"]),
                ("Frequências históricas", dados["baselines_teste"]["frequencias"])]:
    linhas.append({"Modelo": nome, "Acurácia": pct(r["acuracia"]), "Acurácia balanceada": pct(r["acuracia_balanceada"]),
                   "Precisão macro": num(r["precisao_macro"]), "Recall macro": num(r["recall_macro"]),
                   "Macro F1": num(r["f1_macro"]), "Log-Loss": num(r["log_loss"]) if nome != "Sempre mandante" else "n/a",
                   "Brier": num(r["brier"]) if nome != "Sempre mandante" else "n/a", "AUC (OvR)": num(r["roc_auc_ovr_macro"])})
display(pd.DataFrame(linhas).style.hide(axis="index"))
print("Execução:", dados["executado_em"], "| partidas de teste:", teste["n"], "| retreino final:", dados["particoes"]["treino_final"]["n"], "partidas")

Modelo,Acurácia,Acurácia balanceada,Precisão macro,Recall macro,Macro F1,Log-Loss,Brier,AUC (OvR)
Regressão Logística,44.48%,37.26%,0.388,0.373,0.360,1.089,0.652,0.559
Sempre mandante,46.96%,33.33%,0.157,0.333,0.213,n/a,n/a,0.500
Frequências históricas,46.96%,33.33%,0.157,0.333,0.213,1.061,0.640,0.500


Execução: 2026-09-21T12:22:57-03:00 | partidas de teste: 362 | retreino final: 1052 partidas


In [4]:
rot = {"home_win": "Vitória do mandante", "draw": "Empate", "away_win": "Vitória do visitante"}
por_classe = pd.DataFrame([{"Classe": rot[c], "Precisão": num(v["precisao"]), "Recall": num(v["recall"]), "F1": num(v["f1"]), "Suporte": v["suporte"],
                            "Previsões do modelo": teste["distribuicao_previsoes"][c]} for c, v in teste["por_classe"].items()])
display(por_classe.style.hide(axis="index"))
matriz = pd.DataFrame(teste["matriz_confusao"], index=[f"Real: {rot[c]}" for c in rot], columns=[f"Previsto: {rot[c]}" for c in rot])
display(matriz)

Classe,Precisão,Recall,F1,Suporte,Previsões do modelo
Vitória do mandante,0.504,0.724,0.594,170,244
Empate,0.357,0.160,0.221,94,42
Vitória do visitante,0.303,0.235,0.264,98,76


,Previsto: Vitória do mandante,Previsto: Empate,Previsto: Vitória do visitante
Real: Vitória do mandante,123,16,31
Real: Empate,57,15,22
Real: Vitória do visitante,64,11,23


## 3. Treino × validação × teste (sobreajuste)

In [5]:
nome = dados["modelo_selecionado"]
final = dados["final"]
comparacao = pd.DataFrame([
    {"Partição": "Treino (2020–2021)", "Acurácia": pct(dados["modelos"][nome]["treino"]["acuracia"]), "Macro F1": num(dados["modelos"][nome]["treino"]["f1_macro"])},
    {"Partição": "Validação (2022)", "Acurácia": pct(dados["modelos"][nome]["validacao"]["acuracia"]), "Macro F1": num(dados["modelos"][nome]["validacao"]["f1_macro"])},
    {"Partição": "Treino final (2020–2022)", "Acurácia": pct(final["treino_final"]["acuracia"]), "Macro F1": num(final["treino_final"]["f1_macro"])},
    {"Partição": "Teste (2023)", "Acurácia": pct(teste["acuracia"]), "Macro F1": num(teste["f1_macro"])},
])
display(comparacao.style.hide(axis="index"))

Partição,Acurácia,Macro F1
Treino (2020–2021),56.31%,0.512
Validação (2022),41.60%,0.370
Treino final (2020–2022),53.42%,0.485
Teste (2023),44.48%,0.360


## 4. Previsões geradas pelo modelo
Oito primeiras partidas elegíveis do teste, em ordem cronológica (não escolhidas pelo acerto). As probabilidades somam 100% antes do arredondamento.

In [6]:
from src.data_analysis import NOME_CURTO
amostra = []
for e in dados["exemplos"]:
    amostra.append({"Confronto": NOME_CURTO.get(e["mandante"], e["mandante"]) + " × " + NOME_CURTO.get(e["visitante"], e["visitante"]),
                    "P(mandante)": f'{e["p_mandante"]:.1%}', "P(empate)": f'{e["p_empate"]:.1%}', "P(visitante)": f'{e["p_visitante"]:.1%}',
                    "Previsão": e["previsto"], "Real": e["real"], "Placar": e["placar"], "Acerto": "Sim" if e["acertou"] else "Não"})
display(pd.DataFrame(amostra).style.hide(axis="index"))
print("Partidas disputadas entre", dados["exemplos"][0]["data"][:10], "e", dados["exemplos"][-1]["data"][:10])

Confronto,P(mandante),P(empate),P(visitante),Previsão,Real,Placar,Acerto
Palmeiras × Cuiabá,54.6%,32.6%,12.8%,Vitória do mandante,Vitória do mandante,2 x 1,Sim
América-MG × Fluminense,39.7%,31.8%,28.5%,Vitória do mandante,Vitória do visitante,0 x 3,Não
Botafogo × São Paulo,13.0%,42.2%,44.8%,Vitória do visitante,Vitória do mandante,2 x 1,Não
Athletico-PR × Goiás,56.5%,23.2%,20.3%,Vitória do mandante,Vitória do mandante,2 x 0,Sim
Fortaleza × Internacional,34.4%,48.1%,17.5%,Empate,Empate,1 x 1,Sim
Flamengo × Coritiba,83.5%,6.6%,9.8%,Vitória do mandante,Vitória do mandante,3 x 0,Sim
Fluminense × Athletico-PR,66.2%,15.9%,17.9%,Vitória do mandante,Vitória do mandante,2 x 0,Sim
Cuiabá × Bragantino,38.8%,40.7%,20.5%,Empate,Empate,1 x 1,Sim


Partidas disputadas entre 2023-04-15 e 2023-04-22


## 5. Interpretação e erros (descritivos, não causais)

In [7]:
imp = pd.DataFrame(dados["importancia_permutacao"][:8])[["rotulo", "aumento_log_loss", "desvio_padrao"]]
imp.columns = ["Atributo embaralhado", "Aumento do log-loss", "Desvio padrão (30 repetições)"]
display(imp.style.hide(axis="index").format({"Aumento do log-loss": "{:.4f}", "Desvio padrão (30 repetições)": "{:.4f}"}))

erros = dados["analise_erros"]
tabela = pd.DataFrame([{"Data": e["data"], "Confronto": f'{e["mandante"]} × {e["visitante"]}', "Placar": e["placar"],
                        "Previsto": rot[e["previsto"]], "Real": rot[e["real"]], "Confiança": f'{e["confianca"]:.1%}'} for e in erros["erros_mais_confiantes"]])
display(Markdown("**Erros em que o modelo estava mais confiante:**"))
display(tabela.style.hide(axis="index"))

Atributo embaralhado,Aumento do log-loss,Desvio padrão (30 repetições)
diferença de aproveitamento por mando,0.0231,0.0074
equipe mandante,0.0228,0.0139
equipe visitante,0.0212,0.0094
aproveitamento do mandante em casa,0.0098,0.0061
saldo de gols por jogo (últimos 5) — mandante,0.0045,0.0034
cartões amarelos por jogo (últimos 5) — visitante,0.0027,0.0017
gols marcados por jogo (últimos 5) — mandante,0.0021,0.0014
diferença de saldo de gols (mandante − visitante),0.0018,0.0025


**Erros em que o modelo estava mais confiante:**

Data,Confronto,Placar,Previsto,Real,Confiança
2023-10-08,Atlético-MG × Coritiba,1 x 2,Vitória do mandante,Vitória do visitante,88.5%
2023-11-08,América-MG × Coritiba,0 x 3,Vitória do mandante,Vitória do visitante,80.1%
2023-10-29,Internacional × Coritiba,3 x 4,Vitória do mandante,Vitória do visitante,78.0%
2023-07-16,Cruzeiro × Coritiba,0 x 0,Vitória do mandante,Empate,73.0%
2023-11-08,Athletico-PR × Fortaleza,1 x 1,Vitória do mandante,Empate,71.8%


## 6. Usando o modelo salvo (`models/model.joblib`)
O mesmo artefato é usado pela API. O confronto abaixo é hipotético: usa o histórico até o último jogo da base.

In [8]:
import json
from src.common import CatalogoEquipes, carregar_partidas
from src.predictor import ServicoPrevisao

partidas = carregar_partidas()
servico = ServicoPrevisao(partidas, CatalogoEquipes(partidas))
resposta = servico.prever("flamengo", "palmeiras")
print("Modelo carregado:", resposta["model"])
print("Referência:", resposta["reference"])
print("Probabilidades:", resposta["probabilities"], "->", resposta["label"])
print("Forma recente (mandante):", resposta["home_team_form"]["last_five"], "| (visitante):", resposta["away_team_form"]["last_five"])

Modelo carregado: {'algorithm': 'Regressão Logística', 'version': '1.0.0', 'trained_at': '2026-09-21T12:22:57-03:00'}
Referência: {'date': '2023-12-07', 'hour': 16, 'weekday': 'quinta-feira', 'source': 'padrao', 'history_until': '2023-12-06', 'hypothetical': True}
Probabilidades: {'home_win': 0.4867, 'draw': 0.1094, 'away_win': 0.4039} -> Vitória do Flamengo
Forma recente (mandante): ['V', 'V', 'D', 'V', 'D'] | (visitante): ['V', 'E', 'V', 'V', 'E']


In [9]:
fmt = lambda v: f"{v:.2%}".replace(".", ",")
display(Markdown(f'''## 7. Leitura crítica

* **Desempenho geral:** {dados["modelo_selecionado"]} obteve acurácia de {fmt(teste["acuracia"])} e Macro F1 de {teste["f1_macro"]:.3f} no teste; a referência que sempre prevê o mandante obteve {fmt(dados["baselines_teste"]["majoritaria"]["acuracia"])} e {dados["baselines_teste"]["majoritaria"]["f1_macro"]:.3f}.
* **Classes difíceis:** o recall foi de {fmt(teste["por_classe"]["draw"]["recall"])} para empates e {fmt(teste["por_classe"]["away_win"]["recall"])} para vitórias do visitante; o modelo previu empate em apenas {teste["distribuicao_previsoes"]["draw"]} de {teste["n"]} partidas (a maior probabilidade de empate atribuída foi {erros["maior_probabilidade_de_empate"]:.1%}).
* **Probabilidades:** o Log-Loss do modelo foi {teste["log_loss"]:.3f}, contra {dados["baselines_teste"]["frequencias"]["log_loss"]:.3f} da referência de frequências históricas; o modelo não é melhor em qualidade probabilística.
* **Generalização:** acurácia de {fmt(final["treino_final"]["acuracia"])} no treino final contra {fmt(teste["acuracia"])} no teste indica sobreajuste moderado.
* **Limites:** as evidências mostram que o pipeline funciona e permitem inspecionar acertos e erros; não demonstram superioridade geral do modelo.

Arquivos gerados: `models/model.joblib`, `models/metadata.json`, `reports/evidencias/metricas_execucao.json` e `reports/evidencias/previsoes_exemplo.json`.'''))

## 7. Leitura crítica

* **Desempenho geral:** Regressão Logística obteve acurácia de 44,48% e Macro F1 de 0.360 no teste; a referência que sempre prevê o mandante obteve 46,96% e 0.213.
* **Classes difíceis:** o recall foi de 15,96% para empates e 23,47% para vitórias do visitante; o modelo previu empate em apenas 42 de 362 partidas (a maior probabilidade de empate atribuída foi 59.3%).
* **Probabilidades:** o Log-Loss do modelo foi 1.089, contra 1.061 da referência de frequências históricas; o modelo não é melhor em qualidade probabilística.
* **Generalização:** acurácia de 53,42% no treino final contra 44,48% no teste indica sobreajuste moderado.
* **Limites:** as evidências mostram que o pipeline funciona e permitem inspecionar acertos e erros; não demonstram superioridade geral do modelo.

Arquivos gerados: `models/model.joblib`, `models/metadata.json`, `reports/evidencias/metricas_execucao.json` e `reports/evidencias/previsoes_exemplo.json`.